<a href="https://colab.research.google.com/github/simon-drury/sfl-meaning-matrix-llm/blob/main/wadapt_lora_training_sketch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sys
import os
import numpy as np
import faiss
import torch

# Clone the repository to ensure we are using the canonical architecture
if not os.path.exists('sfl-meaning-matrix-llm'):
    !git clone https://github.com/simon-drury/sfl-meaning-matrix-llm.git

repo_path = os.path.abspath('sfl-meaning-matrix-llm')
if repo_path not in sys.path:
    sys.path.append(repo_path)

# Inspect the repository engine to verify the 3x3 (9D) mapping of metafunctions to Field, Tenor, and Mode
path = 'sfl-meaning-matrix-llm/sfl_matrix_engine.py'
with open(path, 'r') as f:
    print("--- Canonical SFLMatrixEngine Source ---")
    print(f.read())

In [ ]:
from sfl_matrix_engine import SFLMatrixEngine

# Initialize the canonical engine
engine = SFLMatrixEngine()
print(f"Engine initialized with {engine.dimension} dimensions.")
# Rows: ideational, interpersonal, textual. Columns: field, tenor, mode.

### Step 2: Populating the Meaning Vector Lexicon (VL)
Using the validated engine logic, we map initial meaning seeds into the 9D substrate and index them with FAISS for high-dimensional retrieval.

In [ ]:
lexicon_seeds = [
    "resource", "process", "structure", "relation", "power",
    "system", "function", "stratification", "rank", "realization"
]

# Generate 9D vectors for the seeds using the repo's logic
vl_matrices = np.array([engine.lexis_to_matrix(item) for item in lexicon_seeds]).astype('float32')

# Add the vectors to the FAISS index
engine.index.add(vl_matrices)

# Store the lexical items for lookup
engine.lexicon.extend(lexicon_seeds)

print(f"Lexical Vector Lexicon (VL) created with {len(engine.lexicon)} items.")
print("First 3 items and their 9D vectors:")
for i in range(min(3, len(engine.lexicon))):
    print(f"  '{engine.lexicon[i]}': {engine.index.reconstruct(i)}")

### Step 3: Generating Meaning Trajectories
We now define the function to map input text into a sequence of 9D vectors. This 'Meaning Trajectory' represents the flow of metafunctional choices in the SFL substrate.

In [ ]:
def get_meaning_trajectory(text, engine):
    """
    Parses input text into a trajectory of 9D vectors using the
    canonical engine's lexis_to_matrix logic.
    """
    words = text.lower().split()
    trajectory = []

    for word in words:
        # Vectorize using the repository-aligned engine
        vector = engine.lexis_to_matrix(word)
        trajectory.append(vector)

    return np.array(trajectory)

# Example usage with SFL-specific terminology
sample_text = "The system realizes meaning through functional structures."
trajectory = get_meaning_trajectory(sample_text, engine)

print(f"Input Text: {sample_text}")
print(f"Trajectory Shape: {trajectory.shape} (Words x 9 Dimensions)")
display(trajectory)

### Step 4: Import and Verify Repository Engine
To ensure strict adherence to your 9D substrate, we will now import the `SFLMatrixEngine` directly from the cloned repository. This replaces the initial 'smoke test' class with your canonical implementation.

In [ ]:
import sys
import os

# Ensure the repository directory is in the python path
repo_path = os.path.abspath('sfl-meaning-matrix-llm')
if repo_path not in sys.path:
    sys.path.append(repo_path)

# Import the actual engine class from the repository
try:
    from sfl_matrix_engine import SFLMatrixEngine as RepoEngine

    # Re-initialize the engine using the repository's logic
    engine = RepoEngine()
    print("Successfully initialized SFLMatrixEngine from repository.")
    print(f"Dimension check: {engine.dimension}")

    # Re-populate the VL index with the seeds to ensure consistency
    vl_matrices = np.array([engine.lexis_to_matrix(item) for item in lexicon_seeds])
    engine.index.add(vl_matrices)
    print(f"Re-synchronized VL with {len(engine.lexicon)} items.")

except ImportError as e:
    print(f"Error importing from repository: {e}")
except AttributeError as e:
    print(f"Engine mismatch: {e}. Checking structure...")

### Step 5: Generating Meaning Trajectories
With the engine verified, we now define a function to map raw text to a sequence of vectors. This represents the 'Meaning Trajectory' in the 9D SFL substrate.

In [ ]:
def get_meaning_trajectory(text, engine):
    """
    Parses input text into a trajectory of 9D vectors.
    In a production setting, this would involve a dependency parser
    or SFL-specific chunker. Here, we map words to vectors.
    """
    # Map each word into a trajectory state
    words = text.lower().split()
    trajectory = []

    for word in words:
        vector = engine.lexis_to_matrix(word)
        trajectory.append(vector)

    return np.array(trajectory)

# Example usage
sample_text = "The system realizes the function through stratification."
trajectory = get_meaning_trajectory(sample_text, engine)

print(f"Input Text: {sample_text}")
print(f"Trajectory Shape: {trajectory.shape} (Words x Dimensions)")
display(trajectory)

### Step 6: Dataset Preparation for Wadapt (LoRA)
To train the adapter, we need a dataset that maps SFL Meaning Trajectories (Source) to the target LLM's expected output (or hidden states). We will define a class to manage this alignment data.

In [ ]:
class SFLAlignmentDataset:
    def __init__(self, engine):
        self.engine = engine
        self.data = []

    def add_entry(self, text):
        """Pairs text with its 9D trajectory."""
        trajectory = get_meaning_trajectory(text, self.engine)
        self.data.append({
            'text': text,
            'trajectory': trajectory
        })

    def get_batch(self):
        return self.data

# Initialize dataset and add example training pairs
setup_data = [
    "Systemic functional linguistics is a functional approach to language.",
    "The interpersonal metafunction enacts social relationships.",
    "The textual metafunction organizes the flow of information.",
    "Ideation represents experience through logical structures."
]

alignment_ds = SFLAlignmentDataset(engine)
for phrase in setup_data:
    alignment_ds.add_entry(phrase)

print(f"Alignment dataset initialized with {len(alignment_ds.data)} training pairs.")
display(alignment_ds.data[0])

### Step 7: Defining the Wadapt (LoRA) Projection Layer
We define an adapter that maps the 9D SFL substrate into a configured model representation space.

In [ ]:
import torch
import torch.nn as nn

class SFLWadapt(nn.Module):
    def __init__(self, sfl_dim=9, llm_hidden_dim=3072, rank=16):
        super(SFLWadapt, self).__init__()
        # Low-rank decomposition (LoRA style) for the mapping
        self.lora_A = nn.Linear(sfl_dim, rank, bias=False)
        self.lora_B = nn.Linear(rank, llm_hidden_dim, bias=False)
        self.scaling = 0.01

    def forward(self, trajectory):
        """
        Maps a 9D trajectory [batch, seq_len, 9]
        to model representation space [batch, seq_len, 3072]
        """
        x = self.lora_A(trajectory)
        x = self.lora_B(x)
        return x * self.scaling

# Initialize the adapter
adapter = SFLWadapt()
print(f"Wadapt initialized with rank 16 mapping: 9 -> 3072 dimensions.")

# Test forward pass with a dummy trajectory from our dataset
if len(alignment_ds.data) > 0:
    test_traj = torch.tensor(alignment_ds.data[0]['trajectory']).unsqueeze(0) # Add batch dim
    with torch.no_grad():
        projected_vectors = adapter(test_traj)
    print(f"Projected Trajectory Shape: {projected_vectors.shape}")

### Step 8: Training the SFL Representation Projection
We optimize Wadapt weights with a simple MSE alignment objective between 9D meaning trajectories and target model representations.

In [ ]:
import torch.optim as optim

# Configuration
learning_rate = 1e-3
epochs = 100
optimizer = optim.Adam(adapter.parameters(), lr=learning_rate)
criterion = nn.MSELoss()

# Convert dataset to tensors (for demonstration, using a single batch)
# In production, we'd use a DataLoader and target representations from the configured model
train_data = alignment_ds.get_batch()

print("Starting alignment training...")
for epoch in range(epochs):
    epoch_loss = 0
    for entry in train_data:
        # Prepare input (Trajectory)
        input_traj = torch.tensor(entry['trajectory']).unsqueeze(0)

        # Target: Placeholder target vectors representing canonical model states
        # A configured model can provide target representations
        target_hidden = torch.randn(1, input_traj.size(1), 3072) * 0.1

        # Forward pass
        optimizer.zero_grad()
        output = adapter(input_traj)

        # Loss & Backprop
        loss = criterion(output, target_hidden)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    if (epoch + 1) % 20 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {epoch_loss/len(train_data):.6f}")

print("Alignment training complete.")

### Step 9: Integrating Register (Field, Tenor, Mode)
We map situational variables (Field, Tenor, Mode) to the 9D substrate. Register provides a contextual bias for each word representation.

In [ ]:
def get_register_biased_trajectory(text, engine, field_bias=1.0, tenor_bias=1.0, mode_bias=1.0):
    """
    Applies Register weights to the 9D (3x3, row-major) substrate dimensions:
    - Field affects the field column (dims 0, 3, 6)
    - Tenor affects the tenor column (dims 1, 4, 7)
    - Mode affects the mode column (dims 2, 5, 8)
    """
    words = text.lower().split()
    trajectory = []

    # Register weight vector for the 9D substrate
    register_weights = torch.tensor([field_bias, tenor_bias, mode_bias] * 3)

    for word in words:
        # Get base vector
        vector = torch.tensor(engine.lexis_to_matrix(word))
        # Apply Register bias
        biased_vector = vector * register_weights
        trajectory.append(biased_vector.numpy())

    return np.array(trajectory)

# Example: High Interpersonal Tenor (e.g., a command or formal interaction)
sample_text = "The system realizes the function."
traj_formal = get_register_biased_trajectory(sample_text, engine, tenor_bias=2.5)

print(f"Trajectory with Tenor (Register) bias applied:")
print(f"Shape: {traj_formal.shape}")
display(traj_formal[0]) # Show the biased vector for the first word